# Phase 8-9 学习指南：回测与预测

**重要警告：** 这是新手最容易翻车的阶段。回测看起来很美好，但充满了陷阱。

---
# Phase 8 — Backtesting

回测 = 用历史数据验证策略

**核心原则：** 回测表现 ≠ 未来收益

回测的目的是**理解策略的特性**，不是证明策略有效。

## 1. 回测的基本流程

```
1. 定义策略规则（买入条件、卖出条件）
       ↓
2. 准备历史数据
       ↓
3. 模拟交易（按规则在历史时间点买卖）
       ↓
4. 计算收益和风险指标
       ↓
5. 分析结果（不要急着相信）
```

## 2. 回测的常见陷阱

### 陷阱 1: Look-Ahead Bias（前视偏差）

用未来的数据做决策。

**错误示例：**
```python
# 错误：用全天最高价判断是否买入
if df['High'].iloc[i] > threshold:  # 最高价要收盘后才知道
    buy_at_open()
```

**正确做法：** 只用当时已知的数据
```python
# 正确：用昨天收盘价判断
if df['Close'].iloc[i-1] > threshold:
    buy_at_open_today()
```

### 陷阱 2: Survivorship Bias（生存偏差）

只测试现在还存在的股票，忽略已经退市的。

**影响：** 回测收益虚高，因为糟糕的股票已经不在样本里了。

**解决：** 使用包含退市股票的全样本数据（需要付费数据）。

### 陷阱 3: Overfitting（过拟合）

参数调到完美适配历史数据，但未来完全失效。

**表现：**
- 回测夏普比率 > 3
- 参数微调一点，收益变化巨大
- 策略逻辑复杂，参数很多

**解决：**
- 参数越少越好
- 用样本外数据验证
- 问自己：这个逻辑在经济上说得通吗？

### 陷阱 4: Transaction Costs（交易成本）

忽略手续费、滑点、买卖价差。

**影响：** 高频策略回测看起来赚钱，实盘亏死。

**解决：** 每次交易扣除成本（如 0.1%）

## 3. 简单策略：均线交叉

**策略规则：**
- 买入：短期均线上穿长期均线（金叉）
- 卖出：短期均线下穿长期均线（死叉）

这是一个经典的趋势跟踪策略。

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def ma_crossover_backtest(ticker, short_window=20, long_window=50, 
                          initial_capital=10000, transaction_cost=0.001):
    """
    均线交叉策略回测
    
    Parameters:
        ticker: 股票代码
        short_window: 短期均线天数
        long_window: 长期均线天数
        initial_capital: 初始资金
        transaction_cost: 交易成本（每次交易的百分比）
    """
    # 下载数据
    df = yf.download(ticker, period="5y", progress=False)
    
    # 计算均线
    df['MA_Short'] = df['Adj Close'].rolling(short_window).mean()
    df['MA_Long'] = df['Adj Close'].rolling(long_window).mean()
    
    # 生成信号
    # 1 = 持有, 0 = 空仓
    df['Signal'] = 0
    df.loc[df['MA_Short'] > df['MA_Long'], 'Signal'] = 1
    
    # 交易信号（信号变化时交易）
    df['Position'] = df['Signal'].diff()
    
    # 计算收益
    df['Returns'] = df['Adj Close'].pct_change()
    df['Strategy_Returns'] = df['Signal'].shift(1) * df['Returns']  # 用昨天的信号
    
    # 扣除交易成本
    trades = df['Position'].abs()
    df['Transaction_Cost'] = trades * transaction_cost
    df['Strategy_Returns'] = df['Strategy_Returns'] - df['Transaction_Cost']
    
    # 计算累计收益
    df['Cumulative_Returns'] = (1 + df['Returns']).cumprod()
    df['Cumulative_Strategy'] = (1 + df['Strategy_Returns']).cumprod()
    
    # 统计
    total_trades = trades.sum()
    final_value = initial_capital * df['Cumulative_Strategy'].iloc[-1]
    buy_hold_value = initial_capital * df['Cumulative_Returns'].iloc[-1]
    
    # 最大回撤
    strategy_dd = (df['Cumulative_Strategy'] / df['Cumulative_Strategy'].cummax() - 1).min()
    bh_dd = (df['Cumulative_Returns'] / df['Cumulative_Returns'].cummax() - 1).min()
    
    print(f"\n{'='*50}")
    print(f"  MA Crossover Backtest: {ticker}")
    print(f"  Short MA: {short_window}, Long MA: {long_window}")
    print(f"{'='*50}")
    print(f"\n总交易次数: {total_trades:.0f}")
    print(f"\n策略收益: {(df['Cumulative_Strategy'].iloc[-1]-1)*100:.2f}%")
    print(f"买入持有: {(df['Cumulative_Returns'].iloc[-1]-1)*100:.2f}%")
    print(f"\n最终价值: ${final_value:,.2f}")
    print(f"买入持有价值: ${buy_hold_value:,.2f}")
    print(f"\n策略最大回撤: {strategy_dd*100:.2f}%")
    print(f"买入持有最大回撤: {bh_dd*100:.2f}%")
    
    # 可视化
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # 价格和均线
    axes[0].plot(df.index, df['Adj Close'], label='Price', alpha=0.7)
    axes[0].plot(df.index, df['MA_Short'], label=f'MA{short_window}', alpha=0.8)
    axes[0].plot(df.index, df['MA_Long'], label=f'MA{long_window}', alpha=0.8)
    
    # 标记买卖点
    buy_signals = df[df['Position'] == 1]
    sell_signals = df[df['Position'] == -1]
    axes[0].scatter(buy_signals.index, buy_signals['Adj Close'], 
                    marker='^', color='green', s=100, label='Buy', zorder=5)
    axes[0].scatter(sell_signals.index, sell_signals['Adj Close'], 
                    marker='v', color='red', s=100, label='Sell', zorder=5)
    
    axes[0].set_title(f'{ticker} Price with MA Signals')
    axes[0].set_ylabel('Price')
    axes[0].legend(loc='upper left')
    axes[0].grid(True, alpha=0.3)
    
    # 累计收益对比
    axes[1].plot(df.index, df['Cumulative_Strategy'], label='Strategy', linewidth=2)
    axes[1].plot(df.index, df['Cumulative_Returns'], label='Buy & Hold', linewidth=2, alpha=0.7)
    axes[1].set_title('Cumulative Returns')
    axes[1].set_ylabel('Cumulative Returns')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return df

# 运行回测
result = ma_crossover_backtest("SPY", short_window=20, long_window=50)

## 4. 策略评估指标

不只是看收益，还要看：

| 指标 | 含义 | 好的标准 |
|------|------|----------|
| 年化收益 | 平均每年赚多少 | > 10% |
| 年化波动 | 风险有多大 | < 20% |
| 夏普比率 | 风险调整收益 | > 1 |
| 最大回撤 | 最坏情况亏多少 | < 20% |
| 胜率 | 盈利交易占比 | > 50% |
| 盈亏比 | 平均盈利/平均亏损 | > 1.5 |
| 交易次数 | 是否过度交易 | 适中 |

In [ ]:
def evaluate_strategy(df, risk_free_rate=0.05):
    """
    策略评估指标计算
    """
    returns = df['Strategy_Returns'].dropna()
    
    # 年化收益
    ann_return = (1 + returns.mean()) ** 252 - 1
    
    # 年化波动
    ann_vol = returns.std() * np.sqrt(252)
    
    # 夏普比率
    sharpe = (ann_return - risk_free_rate) / ann_vol
    
    # 最大回撤
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max
    max_dd = drawdown.min()
    
    # 胜率
    winning_days = returns[returns > 0]
    losing_days = returns[returns < 0]
    win_rate = len(winning_days) / len(returns[returns != 0])
    
    # 盈亏比
    avg_win = winning_days.mean() if len(winning_days) > 0 else 0
    avg_loss = abs(losing_days.mean()) if len(losing_days) > 0 else 0
    profit_loss_ratio = avg_win / avg_loss if avg_loss > 0 else 0
    
    print(f"\n策略评估指标:")
    print(f"  年化收益: {ann_return*100:.2f}%")
    print(f"  年化波动: {ann_vol*100:.2f}%")
    print(f"  夏普比率: {sharpe:.2f}")
    print(f"  最大回撤: {max_dd*100:.2f}%")
    print(f"  胜率: {win_rate*100:.1f}%")
    print(f"  盈亏比: {profit_loss_ratio:.2f}")
    
    return {
        'ann_return': ann_return,
        'ann_vol': ann_vol,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'win_rate': win_rate,
        'profit_loss_ratio': profit_loss_ratio
    }

metrics = evaluate_strategy(result)

## 5. 参数敏感性测试

测试不同参数下策略表现是否稳定。

如果参数稍微变一点，收益变化巨大，说明过拟合风险高。

In [ ]:
def parameter_sensitivity(ticker, short_range=(10, 30), long_range=(40, 80)):
    """
    测试不同参数组合的策略表现
    """
    results = []
    
    for short in range(short_range[0], short_range[1] + 1, 5):
        for long in range(long_range[0], long_range[1] + 1, 5):
            if short >= long:
                continue
            
            # 下载数据
            df = yf.download(ticker, period="5y", progress=False)
            
            # 计算均线
            df['MA_Short'] = df['Adj Close'].rolling(short).mean()
            df['MA_Long'] = df['Adj Close'].rolling(long).mean()
            
            # 信号
            df['Signal'] = 0
            df.loc[df['MA_Short'] > df['MA_Long'], 'Signal'] = 1
            
            # 收益
            df['Returns'] = df['Adj Close'].pct_change()
            df['Strategy_Returns'] = df['Signal'].shift(1) * df['Returns']
            
            # 计算夏普
            returns = df['Strategy_Returns'].dropna()
            ann_return = (1 + returns.mean()) ** 252 - 1
            ann_vol = returns.std() * np.sqrt(252)
            sharpe = (ann_return - 0.05) / ann_vol if ann_vol > 0 else 0
            
            results.append({
                'Short': short,
                'Long': long,
                'Sharpe': sharpe,
                'Ann_Return': ann_return * 100
            })
    
    return pd.DataFrame(results)

# 运行
sensitivity_df = parameter_sensitivity("SPY")

print("参数敏感性测试结果（按夏普排序）:")
print(sensitivity_df.sort_values('Sharpe', ascending=False).head(10).to_string(index=False))

# 热力图
pivot = sensitivity_df.pivot(index='Short', columns='Long', values='Sharpe')

plt.figure(figsize=(10, 6))
import seaborn as sns
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Sharpe Ratio Heatmap (MA Crossover Parameters)')
plt.show()

print("\n注意：如果热力图变化剧烈，说明参数敏感，过拟合风险高")
print("稳定的策略应该有大片相似的参数区域")

## 自测问题（Phase 8）

1. 什么是 Look-Ahead Bias？举一个例子。

2. 为什么说回测夏普比率 > 3 通常不是好事？

3. 如果你的策略在 SPY 上回测很好，但在 QQQ 上回测很差，说明了什么？

4. 交易成本对高频策略和低频策略的影响有什么不同？

5. 如何判断你的策略是否过拟合？

---
# Phase 9 — Prediction System

**核心原则：** 不预测「明天涨跌」，而是预测「概率分布」。

预测系统的作用是**辅助决策**，不是代替判断。

## 1. 预测目标的设定

### 错误目标
```
预测：明天涨还是跌？
问题：噪声太大，准确率很难超过 55%
```

### 正确目标
```
预测：未来 20 天收益率落入哪个区间？
      未来 5 天是否会跑赢 SPY？
      未来 20 天是否会有大波动？
```

**为什么？**
- 时间框架长一点，噪声少一点
- 概率预测比方向预测更有用
- 相对收益比绝对收益更容易预测

## 2. 特征工程

特征 = 模型的输入

好的特征比复杂的模型更重要。

### 2.1 特征类型

| 类型 | 示例 | 信息 |
|------|------|------|
| **价格特征** | 过去 N 天收益率 | 动量 |
| **波动特征** | 过去 N 天波动率 | 风险 |
| **技术指标** | RSI, MACD, MA 距离 | 技术状态 |
| **成交量特征** | 成交量变化 | 市场参与度 |
| **相对特征** | vs SPY 表现 | 相对强度 |

In [ ]:
def create_features(df):
    """
    创建预测特征
    
    注意：只能用当时已知的数据
    """
    df = df.copy()
    
    # 收益率特征
    df['Return_5d'] = df['Adj Close'].pct_change(5)
    df['Return_20d'] = df['Adj Close'].pct_change(20)
    
    # 滚动统计
    df['Volatility_20d'] = df['Adj Close'].pct_change().rolling(20).std()
    df['Return_Mean_20d'] = df['Adj Close'].pct_change().rolling(20).mean()
    
    # RSI
    delta = df['Adj Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / loss))
    
    # MA 距离
    df['MA_20'] = df['Adj Close'].rolling(20).mean()
    df['MA_50'] = df['Adj Close'].rolling(50).mean()
    df['MA_Dist_20'] = (df['Adj Close'] - df['MA_20']) / df['MA_20']
    df['MA_Dist_50'] = (df['Adj Close'] - df['MA_50']) / df['MA_50']
    
    # 成交量变化
    df['Volume_Change'] = df['Volume'].pct_change(5)
    
    return df

# 测试
spy = yf.download("SPY", period="5y", progress=False)
features_df = create_features(spy)

print("特征示例:")
print(features_df[['Adj Close', 'Return_5d', 'Return_20d', 'RSI', 'MA_Dist_20']].tail())

### 2.2 数据泄露检查

**最致命的错误：** 用未来数据创建特征

**检查方法：**
```python
# 正确：用 shift 确保只用过去数据
df['Feature'] = some_calculation.shift(1)  # 昨天的数据

# 错误：直接用
df['Feature'] = some_calculation  # 包含今天的数据
```

## 3. 目标变量创建

定义你要预测什么。

In [ ]:
def create_target(df, horizon=20, threshold=0.02):
    """
    创建目标变量
    
    Parameters:
        horizon: 预测未来多少天
        threshold: 什么算"大幅上涨"
    """
    df = df.copy()
    
    # 未来 N 天收益率
    df['Future_Return'] = df['Adj Close'].shift(-horizon) / df['Adj Close'] - 1
    
    # 分类目标：未来涨跌
    # 1 = 大涨 (>threshold), 0 = 平盘, -1 = 大跌 (<-threshold)
    df['Target_Class'] = 0
    df.loc[df['Future_Return'] > threshold, 'Target_Class'] = 1
    df.loc[df['Future_Return'] < -threshold, 'Target_Class'] = -1
    
    # 二分类目标：是否跑赢 SPY（需要单独计算）
    # 这里简化为是否上涨
    df['Target_Up'] = (df['Future_Return'] > 0).astype(int)
    
    return df

# 测试
df = create_features(spy)
df = create_target(df, horizon=20)

print("\n目标变量分布:")
print(df['Target_Class'].value_counts().sort_index())

## 4. 时间序列划分

**绝对不能用随机划分！**

时间序列必须按时间顺序划分：训练集 → 验证集 → 测试集

In [ ]:
def time_series_split(df, train_ratio=0.6, val_ratio=0.2):
    """
    时间序列数据划分
    
    必须：训练 → 验证 → 测试（按时间顺序）
    """
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train = df.iloc[:train_end]
    val = df.iloc[train_end:val_end]
    test = df.iloc[val_end:]
    
    print(f"数据划分:")
    print(f"  训练集: {train.index[0].date()} ~ {train.index[-1].date()} ({len(train)} 天)")
    print(f"  验证集: {val.index[0].date()} ~ {val.index[-1].date()} ({len(val)} 天)")
    print(f"  测试集: {test.index[0].date()} ~ {test.index[-1].date()} ({len(test)} 天)")
    
    return train, val, test

# 划分
train, val, test = time_series_split(df.dropna())

## 5. 基础模型训练

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 特征列
feature_cols = ['Return_5d', 'Return_20d', 'Volatility_20d', 'RSI', 
                'MA_Dist_20', 'MA_Dist_50', 'Volume_Change']

# 准备数据
train_clean = train.dropna(subset=feature_cols + ['Target_Up'])
val_clean = val.dropna(subset=feature_cols + ['Target_Up'])

X_train = train_clean[feature_cols]
y_train = train_clean['Target_Up']
X_val = val_clean[feature_cols]
y_val = val_clean['Target_Up']

# 训练模型
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]  # 概率

# 评估
print("\n验证集评估:")
print(f"准确率: {accuracy_score(y_val, y_pred):.2%}")
print(f"\n分类报告:\n{classification_report(y_val, y_pred)}")

## 6. 特征重要性分析

In [ ]:
# 特征重要性
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(importance['Feature'], importance['Importance'])
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\n解读：")
print("- 重要性高的特征对预测贡献大")
print("- 可以尝试去掉不重要的特征简化模型")

## 7. Walk-Forward 验证

更严谨的验证方式：滚动窗口训练和测试。

In [ ]:
def walk_forward_validation(df, feature_cols, target_col, 
                            train_window=252, test_window=63, step=21):
    """
    Walk-Forward 验证
    
    模拟真实情况：每次用固定窗口训练，预测未来一段时间
    """
    results = []
    df_clean = df.dropna(subset=feature_cols + [target_col])
    
    for i in range(train_window, len(df_clean) - test_window, step):
        # 训练窗口
        train = df_clean.iloc[i - train_window:i]
        # 测试窗口
        test = df_clean.iloc[i:i + test_window]
        
        # 训练
        X_train = train[feature_cols]
        y_train = train[target_col]
        X_test = test[feature_cols]
        y_test = test[target_col]
        
        model = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
        model.fit(X_train, y_train)
        
        # 预测
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        
        results.append({
            'Date': df_clean.index[i],
            'Accuracy': acc,
            'Samples': len(test)
        })
    
    return pd.DataFrame(results)

# 运行
wf_results = walk_forward_validation(df, feature_cols, 'Target_Up')

print(f"\nWalk-Forward 验证结果:")
print(f"  平均准确率: {wf_results['Accuracy'].mean():.2%}")
print(f"  准确率标准差: {wf_results['Accuracy'].std():.2%}")

# 可视化
plt.figure(figsize=(12, 4))
plt.plot(wf_results['Date'], wf_results['Accuracy'], marker='o')
plt.axhline(y=0.5, color='red', linestyle='--', label='Random Guess')
plt.title('Walk-Forward Validation Accuracy')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n解读：")
print("- 如果准确率稳定在 0.5 以上，说明模型有一定预测能力")
print("- 准确率波动大说明模型不稳定")

## 8. 模型局限性

**模型做不到的：**
- 预测黑天鹅事件
- 预测政策突变
- 长期准确预测

**模型能做到的：**
- 捕捉统计规律
- 辅助风险判断
- 提供概率估计

**使用建议：**
1. 模型输出作为参考，不作为决策依据
2. 关注概率而非分类结果
3. 结合其他分析方法
4. 定期重新训练模型

## 自测问题（Phase 9）

1. 为什么不预测「明天涨跌」？

2. 什么是数据泄露？在特征工程中如何避免？

3. 为什么时间序列不能用随机划分训练/测试集？

4. Walk-Forward 验证和普通验证有什么区别？

5. 如果模型准确率只有 52%，这个模型有用吗？

6. 你会如何在实际中使用预测模型？

---
## 学习检查清单

### Phase 8
- [ ] 理解回测的基本流程
- [ ] 认识并避免 Look-Ahead Bias
- [ ] 认识 Survivorship Bias
- [ ] 理解过拟合的风险
- [ ] 考虑交易成本
- [ ] 能进行参数敏感性测试

### Phase 9
- [ ] 理解预测目标的正确设定
- [ ] 能进行特征工程
- [ ] 避免数据泄露
- [ ] 使用时间序列划分
- [ ] 理解 Walk-Forward 验证
- [ ] 认识模型局限性

---
**完成后，切换到 Codex 执行 Phase 8-9 的代码实现。**